# 06 — CNN Model Comparison for Visual Retrieval

This notebook compares different pretrained CNN backbones for visual product
retrieval.

## Objective

The current retrieval system uses a pretrained ResNet-50 as a frozen feature
extractor. In this notebook, we investigate whether alternative CNN
architectures provide a better trade-off between:

- embedding dimensionality
- retrieval quality
- feature extraction time
- model size / computational cost

## Models Compared

1. ResNet-18
2. ResNet-50
3. EfficientNet-B0

All models use pretrained ImageNet weights and are evaluated as frozen
feature extractors.

## Experimental Pipeline

```text
Product Images
      ↓
Pretrained CNN
      ↓
Image Embeddings
      ↓
L2 Normalization
      ↓
FAISS IndexFlatIP
      ↓
Top-K Retrieval
      ↓
Recall@K / Precision@K / mAP

## 1. Imports

In [1]:
import time
import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import faiss

from PIL import Image
from torch.utils.data import DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm

d:\Projects\VisualProductSearch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Project Paths and Device

The CNN models will run on the available GPU when CUDA is available.

FAISS will be used with `IndexFlatIP` on CPU, matching the retrieval setup
used in Notebook 4 and Notebook 5.

In [2]:
PROJECT_ROOT = Path.cwd().parent

if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path.cwd()

METADATA_PATH = PROJECT_ROOT / "data" / "processed" / "product_metadata.csv"
IMAGE_ROOT = PROJECT_ROOT / "data" / "fashion-product-images-small" / "images"

print("Project root:", PROJECT_ROOT)
print("Metadata exists:", METADATA_PATH.exists())
print("Image directory exists:", IMAGE_ROOT.exists())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Project root: d:\Projects\VisualProductSearch
Metadata exists: True
Image directory exists: True
Device: cuda
GPU: NVIDIA GeForce RTX 2050


## 3. Load Product Metadata

The metadata contains the cleaned set of products with available images.

We use the same catalog as the previous notebooks so that model comparisons
are performed on the same data.

In [3]:
metadata = pd.read_csv(METADATA_PATH)

print("Products:", len(metadata))
print("Columns:", metadata.columns.tolist())

display(metadata.head())

Products: 44441
Columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image_path']


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_path
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt,d:\Projects\VisualProductSearch\data\fashion-p...
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans,d:\Projects\VisualProductSearch\data\fashion-p...
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch,d:\Projects\VisualProductSearch\data\fashion-p...
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants,d:\Projects\VisualProductSearch\data\fashion-p...
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt,d:\Projects\VisualProductSearch\data\fashion-p...


## 4. Image Preprocessing

All models receive the same input preprocessing.

The pretrained ImageNet models expect:

- 224 × 224 image input
- RGB images
- ImageNet normalization

Using identical preprocessing keeps the comparison fair.

In [4]:
image_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## 5. Dataset for Model Comparison

The same images will be passed through every CNN.

The dataset returns:

```text
image tensor
product ID

In [5]:
class ProductImageDataset(torch.utils.data.Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]

        image_path = PROJECT_ROOT / row["image_path"]

        image = Image.open(image_path).convert("RGB")
        image = self.transform(image)

        product_id = int(row["id"])

        return image, product_id

In [6]:
comparison_dataset = ProductImageDataset(
    metadata,
    image_transform
)

comparison_loader = DataLoader(
    comparison_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

sample_images, sample_ids = next(iter(comparison_loader))

print("Batch shape:", sample_images.shape)
print("First product IDs:", sample_ids[:5].tolist())

Batch shape: torch.Size([32, 3, 224, 224])
First product IDs: [15970, 39386, 59263, 21379, 53759]


## 6. Load Pretrained CNN Backbones

We compare three pretrained architectures:

- ResNet-18
- ResNet-50
- EfficientNet-B0

The final classification layer is removed/replaced with `nn.Identity()` so
that the network outputs feature vectors instead of ImageNet class scores.

All models remain frozen.

This means no model is trained on the fashion dataset in this experiment.

In [7]:
models_dict = {}

# ResNet-18
resnet18_weights = models.ResNet18_Weights.DEFAULT
resnet18 = models.resnet18(weights=resnet18_weights)
resnet18.fc = nn.Identity()

models_dict["ResNet-18"] = resnet18


# ResNet-50
resnet50_weights = models.ResNet50_Weights.DEFAULT
resnet50 = models.resnet50(weights=resnet50_weights)
resnet50.fc = nn.Identity()

models_dict["ResNet-50"] = resnet50


# EfficientNet-B0
efficientnet_weights = models.EfficientNet_B0_Weights.DEFAULT
efficientnet = models.efficientnet_b0(weights=efficientnet_weights)
efficientnet.classifier[1] = nn.Identity()

models_dict["EfficientNet-B0"] = efficientnet

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\ANSHIKA/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:07<00:00, 5.97MB/s]


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\ANSHIKA/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:03<00:00, 6.27MB/s]


In [8]:
for name, model in models_dict.items():
    model = model.to(device)
    model.eval()

    for parameter in model.parameters():
        parameter.requires_grad = False

    models_dict[name] = model

    print(f"{name}: ready")

ResNet-18: ready
ResNet-50: ready
EfficientNet-B0: ready


## 7. Verify Embedding Dimensions

Before generating the full catalog embeddings, verify the output dimension
of each architecture using one batch.

Expected dimensions:

```text
ResNet-18       → 512
ResNet-50       → 2048
EfficientNet-B0 → 1280

In [9]:
embedding_dimensions = {}

with torch.no_grad():
    sample_batch = sample_images.to(device)

    for name, model in models_dict.items():
        output = model(sample_batch)

        embedding_dimensions[name] = output.shape[1]

        print(
            f"{name}: "
            f"shape={tuple(output.shape)}, "
            f"embedding_dim={output.shape[1]}"
        )

ResNet-18: shape=(32, 512), embedding_dim=512
ResNet-50: shape=(32, 2048), embedding_dim=2048
EfficientNet-B0: shape=(32, 1280), embedding_dim=1280


## 8. Generate Embeddings and Measure Inference Time

Each model will process the complete product catalog.

We measure:

- total feature extraction time
- number of products
- embedding dimensionality

The same DataLoader and catalog order are used for every model.

Because `shuffle=False`, the embedding row order remains aligned with the
product metadata.

In [10]:
def generate_embeddings(model, data_loader, device):
    model.eval()

    all_embeddings = []
    all_ids = []

    if device.type == "cuda":
        torch.cuda.synchronize()

    start_time = time.perf_counter()

    with torch.no_grad():
        for images, product_ids in tqdm(
            data_loader,
            desc="Generating embeddings"
        ):
            images = images.to(device, non_blocking=True)

            features = model(images)

            features = features.cpu().numpy().astype(np.float32)

            all_embeddings.append(features)
            all_ids.append(product_ids.numpy())

    if device.type == "cuda":
        torch.cuda.synchronize()

    elapsed_time = time.perf_counter() - start_time

    embeddings = np.vstack(all_embeddings)
    product_ids = np.concatenate(all_ids)

    return embeddings, product_ids, elapsed_time

## 9. Run the Model Comparison

This cell generates embeddings for the complete 44,441-product catalog for
each backbone.

This may take several minutes depending on GPU performance.

In [11]:
model_embeddings = {}
model_ids = {}
inference_times = {}

for name, model in models_dict.items():

    print("\n" + "=" * 70)
    print(f"Running: {name}")
    print("=" * 70)

    embeddings_model, ids_model, elapsed = generate_embeddings(
        model,
        comparison_loader,
        device
    )

    model_embeddings[name] = embeddings_model
    model_ids[name] = ids_model
    inference_times[name] = elapsed

    print(f"Embedding shape: {embeddings_model.shape}")
    print(f"Inference time: {elapsed:.2f} seconds")


Running: ResNet-18


Generating embeddings: 100%|██████████| 1389/1389 [10:11<00:00,  2.27it/s]


Embedding shape: (44441, 512)
Inference time: 611.16 seconds

Running: ResNet-50


Generating embeddings: 100%|██████████| 1389/1389 [12:13<00:00,  1.89it/s]


Embedding shape: (44441, 2048)
Inference time: 733.85 seconds

Running: EfficientNet-B0


Generating embeddings: 100%|██████████| 1389/1389 [05:25<00:00,  4.27it/s]


Embedding shape: (44441, 1280)
Inference time: 325.05 seconds


## 10. Validate Embedding Alignment

Every model must produce:

```text
44,441 embedding rows

In [13]:
expected_ids = metadata["id"].to_numpy()

for name in models_dict:

    embeddings_model = model_embeddings[name]
    ids_model = model_ids[name]

    assert embeddings_model.shape[0] == len(metadata)

    assert np.array_equal(
        ids_model,
        expected_ids
    )

    assert embeddings_model.dtype == np.float32

    print(
        f"✓ {name}: "
        f"{embeddings_model.shape[0]} products aligned correctly"
    )

✓ ResNet-18: 44441 products aligned correctly
✓ ResNet-50: 44441 products aligned correctly
✓ EfficientNet-B0: 44441 products aligned correctly


## 11. Normalize Embeddings

We use L2 normalization before FAISS retrieval.

For normalized vectors:

```text
inner product ≈ cosine similarity

In [15]:
# Normalize all model embeddings for cosine similarity / FAISS Inner Product

normalized_embeddings = {}

for name, embeddings_model in model_embeddings.items():

    normalized = embeddings_model.copy()

    faiss.normalize_L2(normalized)

    normalized_embeddings[name] = normalized

    norms = np.linalg.norm(normalized, axis=1)

    print(
        f"{name}: "
        f"mean norm={norms.mean():.6f}, "
        f"min={norms.min():.6f}, "
        f"max={norms.max():.6f}"
    )

ResNet-18: mean norm=1.000000, min=1.000000, max=1.000000
ResNet-50: mean norm=1.000000, min=1.000000, max=1.000000
EfficientNet-B0: mean norm=1.000000, min=1.000000, max=1.000000


## 12. Build FAISS Indices

For each model we create an exact FAISS `IndexFlatIP`.

The comparison therefore changes only the CNN backbone.

The retrieval algorithm, similarity measure, catalog, and query set remain
the same.

In [16]:
faiss_indices = {}

for name, embeddings_model in normalized_embeddings.items():

    dimension = embeddings_model.shape[1]

    index = faiss.IndexFlatIP(dimension)
    index.add(embeddings_model)

    faiss_indices[name] = index

    print(
        f"{name}: "
        f"dimension={dimension}, "
        f"vectors={index.ntotal}"
    )

ResNet-18: dimension=512, vectors=44441
ResNet-50: dimension=2048, vectors=44441
EfficientNet-B0: dimension=1280, vectors=44441


## 13. Use the Same Evaluation Queries as Notebook 5

To make the model comparison scientifically meaningful, all models should
be evaluated on the same query products.

We therefore recreate the same sampling procedure:

- random seed = 42
- maximum 1,000 queries
- only article types with at least 11 products

The query product itself will be excluded from retrieval.

In [17]:
RANDOM_SEED = 42
MAX_QUERIES = 1000
K_VALUES = [5, 10]

np.random.seed(RANDOM_SEED)

article_type_counts = (
    metadata["articleType"]
    .value_counts()
)

eligible_article_types = article_type_counts[
    article_type_counts >= 11
].index

eligible_metadata = metadata[
    metadata["articleType"].isin(eligible_article_types)
].copy()

query_count = min(MAX_QUERIES, len(eligible_metadata))

query_indices = np.random.choice(
    eligible_metadata.index.to_numpy(),
    size=query_count,
    replace=False
)

query_indices = np.sort(query_indices)

print("Eligible products:", len(eligible_metadata))
print("Evaluation queries:", len(query_indices))

Eligible products: 44287
Evaluation queries: 1000


## 14. Retrieval and Evaluation Functions

We now use the same relevance definition as Notebook 5.

A retrieved product is considered relevant when its `articleType` matches the
query product's `articleType`.

The query product itself is removed from the retrieval results.

In [18]:
id_to_metadata = (
    metadata
    .set_index("id")
)

id_to_article_type = (
    metadata
    .set_index("id")["articleType"]
    .to_dict()
)


def retrieve_for_model(
    index,
    embeddings,
    query_index,
    k
):
    query_vector = embeddings[
        query_index:query_index + 1
    ].copy()

    faiss.normalize_L2(query_vector)

    scores, indices = index.search(
        query_vector,
        k + 1
    )

    scores = scores[0]
    indices = indices[0]

    mask = indices != query_index

    filtered_indices = indices[mask][:k]
    filtered_scores = scores[mask][:k]

    return filtered_indices, filtered_scores

In [19]:
def recall_at_k(retrieved_ids, relevant_ids):
    relevant_ids = set(relevant_ids)

    if len(relevant_ids) == 0:
        return 0.0

    retrieved_relevant = sum(
        1 for product_id in retrieved_ids
        if product_id in relevant_ids
    )

    return retrieved_relevant / len(relevant_ids)


def precision_at_k(retrieved_ids, relevant_ids):
    relevant_ids = set(relevant_ids)

    if len(retrieved_ids) == 0:
        return 0.0

    retrieved_relevant = sum(
        1 for product_id in retrieved_ids
        if product_id in relevant_ids
    )

    return retrieved_relevant / len(retrieved_ids)


def average_precision(retrieved_ids, relevant_ids, k):
    relevant_ids = set(relevant_ids)

    if len(relevant_ids) == 0:
        return 0.0

    hits = 0
    precision_sum = 0.0

    for rank, product_id in enumerate(
        retrieved_ids[:k],
        start=1
    ):
        if product_id in relevant_ids:
            hits += 1
            precision_sum += hits / rank

    return precision_sum / min(len(relevant_ids), k)

## 15. Evaluate Every Model

Each model is evaluated using exactly the same:

- query products
- relevance criterion
- K values
- FAISS search method

This isolates the effect of the CNN representation.

In [20]:
comparison_results = []

for model_name in models_dict:

    print(f"\nEvaluating {model_name}...")

    index = faiss_indices[model_name]
    embeddings_model = normalized_embeddings[model_name]

    for query_index in tqdm(
        query_indices,
        desc=model_name
    ):

        query_product_id = int(
            metadata.iloc[query_index]["id"]
        )

        query_article_type = id_to_article_type[
            query_product_id
        ]

        relevant_ids = metadata.loc[
            metadata["articleType"] == query_article_type,
            "id"
        ].astype(int).tolist()

        relevant_ids = [
            product_id
            for product_id in relevant_ids
            if product_id != query_product_id
        ]

        row = {
            "model": model_name,
            "query_product_id": query_product_id,
            "article_type": query_article_type,
        }

        for k in K_VALUES:

            retrieved_indices, retrieved_scores = retrieve_for_model(
                index,
                embeddings_model,
                query_index,
                k
            )

            retrieved_ids = [
                int(metadata.iloc[idx]["id"])
                for idx in retrieved_indices
            ]

            row[f"recall@{k}"] = recall_at_k(
                retrieved_ids,
                relevant_ids
            )

            row[f"precision@{k}"] = precision_at_k(
                retrieved_ids,
                relevant_ids
            )

            row[f"ap@{k}"] = average_precision(
                retrieved_ids,
                relevant_ids,
                k
            )

        comparison_results.append(row)

comparison_df = pd.DataFrame(comparison_results)

print("\nEvaluation complete.")
print("Rows:", len(comparison_df))

display(comparison_df.head())


Evaluating ResNet-18...


ResNet-18: 100%|██████████| 1000/1000 [00:17<00:00, 58.03it/s]



Evaluating ResNet-50...


ResNet-50: 100%|██████████| 1000/1000 [00:54<00:00, 18.39it/s]



Evaluating EfficientNet-B0...


EfficientNet-B0: 100%|██████████| 1000/1000 [00:37<00:00, 26.83it/s]



Evaluation complete.
Rows: 3000


,model,query_product_id,article_type,recall@5,precision@5,ap@5,recall@10,precision@10,ap@10
0,ResNet-18,28032,Briefs,0.005896,1.0,1.000000,0.011792,1.0,1.000000
1,ResNet-18,35322,Earrings,0.012019,1.0,1.000000,0.024038,1.0,1.000000
2,ResNet-18,13284,Shorts,0.009158,1.0,1.000000,0.018315,1.0,1.000000
3,ResNet-18,26197,Perfume and Body Mist,0.004894,0.6,0.453333,0.004894,0.3,0.226667
4,ResNet-18,3391,Handbags,0.002844,1.0,1.000000,0.005688,1.0,1.000000


## 16. Overall Model Performance

We aggregate the query-level results to obtain the mean retrieval performance
for each CNN backbone.

In [21]:
overall_metrics = (
    comparison_df
    .groupby("model")
    .agg(
        queries=("query_product_id", "count"),
        recall_at_5=("recall@5", "mean"),
        recall_at_10=("recall@10", "mean"),
        precision_at_5=("precision@5", "mean"),
        precision_at_10=("precision@10", "mean"),
        map_at_5=("ap@5", "mean"),
        map_at_10=("ap@10", "mean"),
    )
)

overall_metrics = overall_metrics.sort_values(
    "map_at_10",
    ascending=False
)

display(overall_metrics)

,queries,recall_at_5,recall_at_10,precision_at_5,precision_at_10,map_at_5,map_at_10
model,,,,,,,
EfficientNet-B0,1000,0.008272,0.015143,0.8006,0.7712,0.759703,0.715680
ResNet-50,1000,0.007771,0.014204,0.7810,0.7538,0.738657,0.695797
ResNet-18,1000,0.007804,0.014172,0.7714,0.7458,0.728447,0.686772


## 17. Add Embedding Dimensions and Inference Time

Retrieval quality alone is not enough.

A model may achieve slightly better retrieval metrics while producing much
larger embeddings or requiring substantially more inference time.

We therefore combine the retrieval metrics with computational information.

In [22]:
model_summary = overall_metrics.copy()

model_summary["embedding_dimension"] = [
    embedding_dimensions[name]
    for name in model_summary.index
]

model_summary["inference_time_seconds"] = [
    inference_times[name]
    for name in model_summary.index
]

model_summary["images_per_second"] = (
    len(metadata)
    / model_summary["inference_time_seconds"]
)

model_summary = model_summary[
    [
        "queries",
        "embedding_dimension",
        "recall_at_5",
        "recall_at_10",
        "precision_at_5",
        "precision_at_10",
        "map_at_5",
        "map_at_10",
        "inference_time_seconds",
        "images_per_second",
    ]
]

display(model_summary)

,queries,embedding_dimension,recall_at_5,recall_at_10,precision_at_5,precision_at_10,map_at_5,map_at_10,inference_time_seconds,images_per_second
model,,,,,,,,,,
EfficientNet-B0,1000,1280,0.008272,0.015143,0.8006,0.7712,0.759703,0.715680,325.048441,136.721160
ResNet-50,1000,2048,0.007771,0.014204,0.7810,0.7538,0.738657,0.695797,733.848901,60.558788
ResNet-18,1000,512,0.007804,0.014172,0.7714,0.7458,0.728447,0.686772,611.162971,72.715466


## 18. Compare Embedding Storage Requirements

Embedding dimensionality directly affects storage requirements.

For float32 embeddings:

```text
storage ≈ number_of_images × embedding_dimension × 4 bytes

In [23]:
storage_rows = []

num_products = len(metadata)

for model_name in models_dict:

    dimension = embedding_dimensions[model_name]

    bytes_required = (
        num_products
        * dimension
        * 4
    )

    storage_mb = bytes_required / (1024 ** 2)

    storage_rows.append({
        "model": model_name,
        "embedding_dimension": dimension,
        "estimated_float32_storage_mb": storage_mb
    })

storage_df = pd.DataFrame(storage_rows)

display(storage_df)

,model,embedding_dimension,estimated_float32_storage_mb
0,ResNet-18,512,86.798828
1,ResNet-50,2048,347.195312
2,EfficientNet-B0,1280,216.997070


## 19. Category-Level Comparison

Overall metrics can hide differences between product categories.

We therefore compare model performance across article types.

Only categories with at least 10 evaluation queries are included to avoid
drawing conclusions from categories represented by very few queries.

In [24]:
category_comparison = (
    comparison_df
    .groupby(["model", "article_type"])
    .agg(
        queries=("query_product_id", "count"),
        recall_at_5=("recall@5", "mean"),
        recall_at_10=("recall@10", "mean"),
        precision_at_5=("precision@5", "mean"),
        precision_at_10=("precision@10", "mean"),
        map_at_5=("ap@5", "mean"),
        map_at_10=("ap@10", "mean"),
    )
    .reset_index()
)

category_comparison = category_comparison[
    category_comparison["queries"] >= 10
]

display(
    category_comparison
    .sort_values(
        ["article_type", "map_at_10"],
        ascending=[True, False]
    )
    .head(30)
)

,model,article_type,queries,recall_at_5,recall_at_10,precision_at_5,precision_at_10,map_at_5,map_at_10
1,EfficientNet-B0,Backpacks,16,0.006656,0.013485,0.962500,0.975000,0.947083,0.953085
85,ResNet-18,Backpacks,16,0.006743,0.013313,0.975000,0.962500,0.971875,0.948891
169,ResNet-50,Backpacks,16,0.006656,0.012794,0.962500,0.925000,0.937292,0.902922
87,ResNet-18,Belts,19,0.006093,0.012121,0.989474,0.984211,0.989474,0.980439
3,EfficientNet-B0,Belts,19,0.006028,0.012121,0.978947,0.984211,0.968596,0.972297
171,ResNet-50,Belts,19,0.006028,0.011991,0.978947,0.973684,0.951930,0.949701
5,EfficientNet-B0,Bra,11,0.010504,0.021008,1.000000,1.000000,1.000000,1.000000
173,ResNet-50,Bra,11,0.010504,0.021008,1.000000,1.000000,1.000000,1.000000
89,ResNet-18,Bra,11,0.010504,0.020817,1.000000,0.990909,1.000000,0.990909
91,ResNet-18,Briefs,19,0.005648,0.011296,0.957895,0.957895,0.957895,0.948181


## 20. Identify the Best Model per Metric

This provides a simple summary of which architecture performs best according
to each retrieval metric.

The goal is not to treat one metric as universally superior, but to inspect
the complete trade-off.

In [25]:
metric_columns = [
    "recall_at_5",
    "recall_at_10",
    "precision_at_5",
    "precision_at_10",
    "map_at_5",
    "map_at_10",
]

best_models = {}

for metric in metric_columns:
    best_model = model_summary[metric].idxmax()
    best_value = model_summary.loc[best_model, metric]

    best_models[metric] = {
        "model": best_model,
        "value": best_value
    }

best_models_df = pd.DataFrame(best_models).T

display(best_models_df)

,model,value
recall_at_5,EfficientNet-B0,0.008272
recall_at_10,EfficientNet-B0,0.015143
precision_at_5,EfficientNet-B0,0.8006
precision_at_10,EfficientNet-B0,0.7712
map_at_5,EfficientNet-B0,0.759703
map_at_10,EfficientNet-B0,0.71568


## 21. Retrieval Quality vs Computational Cost

The final model should not be selected solely because it has the highest
retrieval score.

We consider:

1. Retrieval quality
2. Embedding dimensionality
3. Feature extraction time
4. Storage requirements

A model that is slightly less accurate but substantially faster and smaller
may be preferable for a production-oriented retrieval system.

Conversely, if a larger model provides a meaningful retrieval improvement,
its additional computational cost may be justified.

In [26]:
comparison_table = model_summary.copy()

comparison_table = comparison_table.sort_values(
    "map_at_10",
    ascending=False
)

display(comparison_table)

,queries,embedding_dimension,recall_at_5,recall_at_10,precision_at_5,precision_at_10,map_at_5,map_at_10,inference_time_seconds,images_per_second
model,,,,,,,,,,
EfficientNet-B0,1000,1280,0.008272,0.015143,0.8006,0.7712,0.759703,0.715680,325.048441,136.721160
ResNet-50,1000,2048,0.007771,0.014204,0.7810,0.7538,0.738657,0.695797,733.848901,60.558788
ResNet-18,1000,512,0.007804,0.014172,0.7714,0.7458,0.728447,0.686772,611.162971,72.715466


## 22. Sanity Check: Self-Retrieval

Before interpreting the comparison results, verify that each model can
retrieve the query product itself with extremely high similarity.

The evaluation procedure later removes the query product from the returned
results.

This is only a retrieval sanity check and is not an evaluation metric.

In [27]:
sanity_query_index = int(query_indices[0])

sanity_rows = []

for model_name in models_dict:

    embeddings_model = normalized_embeddings[model_name]
    index = faiss_indices[model_name]

    query_vector = embeddings_model[
        sanity_query_index:sanity_query_index + 1
    ].copy()

    scores, indices = index.search(
        query_vector,
        1
    )

    retrieved_index = int(indices[0][0])
    score = float(scores[0][0])

    sanity_rows.append({
        "model": model_name,
        "query_product_id": int(
            metadata.iloc[sanity_query_index]["id"]
        ),
        "retrieved_product_id": int(
            metadata.iloc[retrieved_index]["id"]
        ),
        "similarity": score,
        "correct_self_retrieval": (
            retrieved_index == sanity_query_index
        )
    })

sanity_df = pd.DataFrame(sanity_rows)

display(sanity_df)

,model,query_product_id,retrieved_product_id,similarity,correct_self_retrieval
0,ResNet-18,28032,28032,1.0,True
1,ResNet-50,28032,28032,1.0,True
2,EfficientNet-B0,28032,28032,1.0,True


## 23. Save Comparison Results

The experiment outputs are saved so that later notebooks and the README can
reference the actual measured results.

Large embedding matrices are not saved by this notebook because they are
generated artifacts and are excluded from GitHub.

In [28]:
results_dir = PROJECT_ROOT / "results"
results_dir.mkdir(exist_ok=True)

comparison_df.to_csv(
    results_dir / "model_comparison_query_results.csv",
    index=False
)

model_summary.to_csv(
    results_dir / "model_comparison_summary.csv"
)

storage_df.to_csv(
    results_dir / "model_embedding_storage.csv",
    index=False
)

category_comparison.to_csv(
    results_dir / "model_comparison_category_metrics.csv",
    index=False
)

print("Saved:")
print(results_dir / "model_comparison_query_results.csv")
print(results_dir / "model_comparison_summary.csv")
print(results_dir / "model_embedding_storage.csv")
print(results_dir / "model_comparison_category_metrics.csv")

Saved:
d:\Projects\VisualProductSearch\results\model_comparison_query_results.csv
d:\Projects\VisualProductSearch\results\model_comparison_summary.csv
d:\Projects\VisualProductSearch\results\model_embedding_storage.csv
d:\Projects\VisualProductSearch\results\model_comparison_category_metrics.csv


## 24. Final Validation

The experiment is considered valid only if:

- all models processed the same catalog
- all models used the same preprocessing
- all models used the same query set
- all embeddings are correctly aligned with product IDs
- embeddings were L2-normalized before cosine-based FAISS retrieval
- the query product was excluded during evaluation
- all models were evaluated using the same relevance definition

In [29]:
assert set(models_dict.keys()) == {
    "ResNet-18",
    "ResNet-50",
    "EfficientNet-B0"
}

for model_name in models_dict:

    assert model_embeddings[model_name].shape[0] == len(metadata)

    assert np.array_equal(
        model_ids[model_name],
        metadata["id"].to_numpy()
    )

    assert faiss_indices[model_name].ntotal == len(metadata)

    assert model_name in inference_times

assert len(comparison_df) == (
    len(query_indices) * len(models_dict)
)

print("✓ All models processed the same catalog")
print("✓ Product ID alignment verified")
print("✓ FAISS index sizes verified")
print("✓ Same evaluation queries used")
print("✓ Model comparison completed successfully")

✓ All models processed the same catalog
✓ Product ID alignment verified
✓ FAISS index sizes verified
✓ Same evaluation queries used
✓ Model comparison completed successfully


# 25. Experiment Summary

This experiment compared three pretrained CNN feature extractors for visual
product retrieval:

- ResNet-18
- ResNet-50
- EfficientNet-B0

All models were used as frozen ImageNet-pretrained feature extractors.

The comparison evaluated:

- embedding dimensionality
- Recall@5
- Recall@10
- Precision@5
- Precision@10
- mAP@5
- mAP@10
- feature extraction time
- estimated embedding storage

The same product catalog, preprocessing pipeline, query set, FAISS retrieval
method, and article-type-based relevance criterion were used across all
models.

The results show the trade-off between retrieval quality and computational
cost.

## Important Limitation

The dataset does not provide human-annotated visual similarity judgments.
Therefore, `articleType` is used as a proxy relevance criterion.

The resulting metrics measure how consistently the models retrieve products
from the same article type. They should not be interpreted as a definitive
measure of human-perceived visual similarity.

## Next Step

The best-performing model should be selected based on both retrieval quality
and computational efficiency.

The next stage is detailed error analysis of retrieval failures and
qualitative inspection of visually similar and incorrect results.